In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Konfiguracja i ładowanie danych

CSV_PATH = "train.csv"
IMG_SIZE = 64  # Rozmiar, do którego przeskalujemy wycięte pola szachownicy
NUM_CLASSES = 13  # Klasy od 0 do 12 zgodnie ze słownikiem types

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Nie znaleziono pliku {CSV_PATH}. Uruchom najpierw skrypt generujący CSV.")

df = pd.read_csv(CSV_PATH)

X = []
y = []

print("Ładowanie i przetwarzanie obrazów...")
for idx, row in df.iterrows():
    img_path = row["image_path"]
    label = row["label"]
    
    if os.path.exists(img_path):
        # Odczyt obrazu (BGR -> RGB)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resizing do stałego rozmiaru wejściowego dla CNN
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        # Normalizacja pikseli do zakresu [0, 1]
        img = img / 255.0
        
        X.append(img)
        y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f"Załadowano pomyślnie {len(X)} obrazów o kształcie {X.shape[1:]}")


# 2. Podział danych (Dokładnie tak jak w oryginale)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


# 3. Definicja architektury CNN

model = models.Sequential([
    # Pierwsza warstwa splotowa + pooling
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D((2, 2)),
    
    # Druga warstwa splotowa + pooling
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Trzecia warstwa splotowa + pooling
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Spłaszczenie macierzy do wektora i warstwy gęste (Dense)
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),  # Regularyzacja zapobiegająca overfittingowi
    layers.Dense(NUM_CLASSES, activation='softmax')  # Softmax dla klasyfikacji wieloklasowej
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


# 4. Trening modelu CNN

EPOCHS = 15
BATCH_SIZE = 32

print("\nRozpoczynanie treningu sieci CNN...")
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)


# 5. Ewaluacja i wyniki akuracji

_, train_acc = model.evaluate(X_train, y_train, verbose=0)
_, val_acc = model.evaluate(X_val, y_val, verbose=0)
_, test_acc = model.evaluate(X_test, y_test, verbose=0)

print("\n" + "="*50)
print("WYNIKI AKURACJI (CNN)")
print("="*50)
print(f"Akuracja train: {train_acc:.3f}")
print(f"Akuracja val:   {val_acc:.3f}")
print(f"Akuracja test:  {test_acc:.3f}")

# Predict (Wyciągamy klasę z najwyższym prawdopodobieństwem za pomocą argmax)
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n" + "="*50)
print("CLASSIFICATION REPORT (test)")
print("="*50)
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:")
print(cm)


# 6. Zapis modelu do pliku (Odpowiednik zapisu wag)

model.save("piece_recognition/model_cnn.h5")
print("\nModel CNN został zapisany do folderu piece_recognition/model_cnn.h5")

# 7. Wizualizacja historii treningu (Krzywe uczenia)

plt.figure(figsize=(12, 5))

# Wykres dokładności (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
plt.xlabel('Epoka')
plt.ylabel('Dokładność')
plt.title('Dokładność modelu w czasie')
plt.legend()
plt.grid(True)

# Wykres straty (Loss)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', marker='s')
plt.xlabel('Epoka')
plt.ylabel('Funkcja straty')
plt.title('Strata modelu w czasie')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("cnn_training_history.png", dpi=150)
plt.close()
print("Wykres historii uczenia został zapisany jako 'cnn_training_history.png'")

I0000 00:00:1781628907.787356   16479 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781628907.843316   16479 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781628908.963557   16479 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Ładowanie i przetwarzanie obrazów...
Załadowano pomyślnie 12672 obrazów o kształcie (64, 64, 3)
Train: 8870, Val: 1901, Test: 1901


/home/krzysztof/Documents/ChessPiececCV/.envCV/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1781628915.503619   16479 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       147,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13)             │           845 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 204,685 (799.55 KB)

 Non-trainable params: 0 (0.00 B)


Rozpoczynanie treningu sieci CNN...
Epoch 1/15


W0000 00:00:1781628915.739142   16479 cpu_allocator_impl.cc:82] Allocation of 435978240 exceeds 10% of free system memory.


278/278 ━━━━━━━━━━━━━━━━━━━━ 17s 56ms/step - accuracy: 0.6849 - loss: 1.3763 - val_accuracy: 0.6881 - val_loss: 1.2318
Epoch 2/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 16s 56ms/step - accuracy: 0.6883 - loss: 1.2731 - val_accuracy: 0.6881 - val_loss: 1.2053
Epoch 3/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step - accuracy: 0.6899 - loss: 1.2509 - val_accuracy: 0.6881 - val_loss: 1.1924
Epoch 4/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 16s 56ms/step - accuracy: 0.6938 - loss: 1.2221 - val_accuracy: 0.6886 - val_loss: 1.1883
Epoch 5/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step - accuracy: 0.6961 - loss: 1.1973 - val_accuracy: 0.7002 - val_loss: 1.1651
Epoch 6/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 16s 57ms/step - accuracy: 0.7002 - loss: 1.1806 - val_accuracy: 0.7049 - val_loss: 1.1369
Epoch 7/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step - accuracy: 0.7035 - loss: 1.1467 - val_accuracy: 0.7028 - val_loss: 1.1287
Epoch 8/15
278/278 ━━━━━━━━━━━━━━━━━━━━ 15s 54ms/step - accuracy: 0.7056 - loss: 1.1255 - val_accurac